# TrojanGYM — Portkey AI gateway version

This notebook ports the **detector-in-the-loop** framework from
*TrojanGYM: A Detector-in-the-Loop LLM for Adaptive RTL Hardware Trojan Insertion*
(arXiv:2601.17178, DfX-NYUAD) onto the **NYU Portkey AI gateway**, replacing the
paper's per-vendor LLM clients (OpenAI / Gemini / Groq / Anthropic) with a single
Portkey backend.

Compared to `ghost_portkey_ai.ipynb` (which is the one-shot GHOST baseline), this
notebook adds the closed loop the paper is about:

1. **Generate** a Trojan-inserted design (LLM, via Portkey).
2. **Validate** it with `iverilog` syntax check **and** a functional testbench,
   feeding any failure back to the LLM for repair (up to `MAX_SYNTAX_RETRIES`).
3. **Detect** it with the Robust-GNN4TJ ensemble.
4. **Evade**: if detected, feed the detection report back and ask the LLM to
   re-insert more stealthily — repeat up to `MAX_EVASION_ATTEMPTS`.

> **Scope.** For authorized hardware-security research only, on designs you are
> permitted to assess. The detection half is defensive: it measures how well a
> GNN detector withstands adaptively generated Trojans.

### What you must provide
* A course Portkey API key. Set `PORTKEY_API_KEY`, or enter it securely when prompted.
* `iverilog` (`brew install icarus-verilog` on macOS) for syntax + testbench checks.
* **Required for the agentic loop**, placed **beside this notebook**: the
  Robust-GNN4TJ assets `dfg_tj_rtl.pkl` and the per-HT model folders
  `model_HT1 … model_HT4`, plus the `hw2vec` (`Robust_hw2vec`) package (cloned
  locally in Section 6) and torch/torch-geometric. If these are not present,
  `run_agentic_loop` raises `DetectorUnavailable` and aborts. (Only the
  generation-only helper `run_generation_pipeline` works without the detector.)


## 1. Install the Portkey client

Run once in a fresh environment. (torch / hw2vec for the optional detector are
installed in Section 6.)

In [ ]:
%pip install -q portkey-ai

## 2. Configuration

Set `PORTKEY_API_KEY` in your environment, or enter it securely when prompted. Everything else can be overridden
by environment variables without editing code. This notebook is **self-contained**:
all working directories and detector assets default to the notebook's own folder
(`NOTEBOOK_DIR`) — nothing is read from an external repo.

In [ ]:
import os
import re
import glob
import shutil
import subprocess
import sys
import time
import json
from pathlib import Path

# Recover Jupyter's IOPub streams if an older notebook run redirected them.
def restore_notebook_streams():
    try:
        shell = get_ipython()
        kernel = getattr(shell, "kernel", None)
        if kernel is not None:
            sys.stdout, sys.stderr = kernel._stdout, kernel._stderr
            # The old redirected stream missed the current cell's parent header.
            parent = shell.get_parent() if hasattr(shell, "get_parent") else None
            if parent:
                for stream in (sys.stdout, sys.stderr):
                    if hasattr(stream, "set_parent"):
                        stream.set_parent(parent)
    except (NameError, AttributeError):
        pass

restore_notebook_streams()

# ── Portkey gateway ────────────────────────────────────────────
PORTKEY_BASE_URL = os.environ.get(
    "PORTKEY_BASE_URL", "https://ai-gateway.apps.cloud.rt.nyu.edu/v1")
PORTKEY_API_KEY = os.environ.get("PORTKEY_API_KEY", "").strip()
if not PORTKEY_API_KEY:
    from getpass import getpass
    PORTKEY_API_KEY = getpass("Enter your course Portkey API key: ").strip()
PORTKEY_MODEL   = os.environ.get("PORTKEY_MODEL", "@vertexai/gemini-2.5-pro")
# Portkey SDK expects request_timeout in milliseconds, not seconds.
PORTKEY_REQUEST_TIMEOUT_MS = int(os.environ.get("PORTKEY_REQUEST_TIMEOUT_MS", "300000"))
# Zero means omit max_tokens and use the provider default. This prevents Gemini
# 2.5 Pro's internal thinking from consuming a fixed output-token budget.
PORTKEY_MAX_TOKENS = int(os.environ.get("PORTKEY_MAX_TOKENS", "0"))

# ── Notebook home — everything is self-contained under here ────
# No dependency on any external repo: detector assets and designs live beside
# this notebook. Override with TJGYM_HOME if you run from a different directory.
_default_home = Path.cwd()
if not (_default_home / "ghost_portkey_trojangym.ipynb").exists():
    _course_candidates = [p.parent for p in _default_home.glob("*/ghost_portkey_trojangym.ipynb")
                          if (p.parent / "README.md").exists()
                          and (p.parent / "dfg_tj_rtl.pkl").exists()]
    if len(_course_candidates) == 1:
        _default_home = _course_candidates[0]
NOTEBOOK_DIR = Path(os.environ.get("TJGYM_HOME", _default_home)).resolve()

# ── Working directories (local, not Colab /content) ────────────
WORK_DIR               = Path(os.environ.get("TJGYM_WORK_DIR", NOTEBOOK_DIR / "trojangym_portkey_runs"))
BASE_DESIGNS_DIR       = Path(os.environ.get("TJGYM_DESIGNS_DIR",   WORK_DIR / "designs"))
VULNERABLE_DESIGNS_DIR = Path(os.environ.get("TJGYM_VULN_DIR",      WORK_DIR / "vulnerable_designs"))
TESTBENCHES_DIR        = Path(os.environ.get("TJGYM_TB_DIR",        WORK_DIR / "testbenches"))

for d in (WORK_DIR, BASE_DESIGNS_DIR, VULNERABLE_DESIGNS_DIR, TESTBENCHES_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── Detector assets (REQUIRED; kept beside this notebook) ──────
# Self-contained: place these next to the notebook (NOTEBOOK_DIR). No external
# repo is referenced.
#   - dfg_tj_rtl.pkl                : graph cache
#   - model_HT1 / ... / model_HT4   : per-HT trained GNN folders (model.cfg + model.pth)
#   - hw2vec/                       : the Robust_hw2vec package (cloned in Section 6)
DATA_PKL_PATH = Path(os.environ.get("TJGYM_DATA_PKL", NOTEBOOK_DIR / "dfg_tj_rtl.pkl"))
HW2VEC_DIR    = Path(os.environ.get("HW2VEC_DIR", NOTEBOOK_DIR / "hw2vec"))
YAML_PATH     = Path(os.environ.get("TJGYM_YAML", HW2VEC_DIR / "examples" / "example_gnn4tj.yaml"))
MODEL_PATHS = {
    "HT1": Path(os.environ.get("TJGYM_MODEL_HT1", NOTEBOOK_DIR / "model_HT1")),
    "HT2": Path(os.environ.get("TJGYM_MODEL_HT2", NOTEBOOK_DIR / "model_HT2")),
    "HT3": Path(os.environ.get("TJGYM_MODEL_HT3", NOTEBOOK_DIR / "model_HT3")),
    "HT4": Path(os.environ.get("TJGYM_MODEL_HT4", NOTEBOOK_DIR / "model_HT4")),
}

# ── Trojan taxonomy & strategies (verbatim from TrojanGYM) ──────
VULNERABILITIES = {
    "T1": "change functionality",
    "T2": "leak information",
    "T3": "denial of service",
    "T4": "performance degradation",
}
INSERTION_STRATEGIES = {
    "T1": "Subtly alter logic to produce incorrect results under specific conditions.",
    "T2": "Design a covert data transmission mechanism that activates upon detecting a specific signal pattern.",
    "T3": "Introduce a condition that temporarily disables the module, triggered by a rare sequence of events.",
    "T4": "Implement a continuously running shift register or accumulator to increase power consumption upon detecting a specific event.",
}
DETECTION_METHODS  = {"D4": "HW2VEC/GNN Analysis"}
EVASION_STRATEGIES = {
    "S4": "Evade HW2VEC by minimizing changes to the Data Flow Graph (DFG) and Abstract Syntax Tree (AST) structure, avoiding new nodes/edges.",
}

# ── Loop control ───────────────────────────────────────────────
MAX_SYNTAX_RETRIES   = int(os.environ.get("TJGYM_MAX_SYNTAX_RETRIES", "7"))
MAX_LLM_CALL_FAILURES = int(os.environ.get("TJGYM_MAX_LLM_CALL_FAILURES", "2"))
MAX_EVASION_ATTEMPTS = int(os.environ.get("TJGYM_MAX_EVASION_ATTEMPTS", "7"))
DFG_TIMEOUT_SECONDS  = int(os.environ.get("TJGYM_DFG_TIMEOUT", "0"))

print("Portkey model :", PORTKEY_MODEL if PORTKEY_API_KEY else "(PORTKEY_API_KEY NOT set)")
print("Request timeout:", f"{PORTKEY_REQUEST_TIMEOUT_MS / 1000:g}s")
print("Output budget  :", PORTKEY_MAX_TOKENS or "provider default")
print("Work dir      :", WORK_DIR)
print("Designs in    :", BASE_DESIGNS_DIR)
print("Vuln out      :", VULNERABLE_DESIGNS_DIR)
print("Detector pkl  :", DATA_PKL_PATH, "(found)" if DATA_PKL_PATH.exists() else "(MISSING — required)")

## 3. LLM backend (Portkey)

A single `get_llm_response(prompt)` used by every stage — initial insertion,
syntax repair, testbench repair, and evasion — so the whole paper pipeline runs
through one gateway. The system prompt is kept verbatim from TrojanGYM to stop the
model truncating multi-module designs.

In [ ]:
from portkey_ai import Portkey

_SYSTEM_PROMPT = (
    "You are an expert in hardware design and verification. "
    "Include ALL code from the input file in your response; you may only ADD code, never remove it. "
    "Do NOT stop generating after the first endmodule. "
    "Ensure ALL modules from the input are preserved. "
    'Do not truncate or use placeholder comments like "rest of code here".'
)

_portkey_client = None

def _get_client():
    global _portkey_client
    if not PORTKEY_API_KEY:
        raise RuntimeError("PORTKEY_API_KEY is empty. Set it, then rerun the config cell.")
    if _portkey_client is None:
        _portkey_client = Portkey(
            base_url=PORTKEY_BASE_URL, api_key=PORTKEY_API_KEY,
            request_timeout=PORTKEY_REQUEST_TIMEOUT_MS)
    return _portkey_client

_LAST_LLM_DIAGNOSTICS = {}

def get_llm_response(prompt, model_id=None):
    """Send a prompt through Portkey. Returns (response_text, model_name)."""
    model_id = model_id or PORTKEY_MODEL
    request = dict(
        model=model_id,
        messages=[
            {"role": "system", "content": _SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
    )
    if PORTKEY_MAX_TOKENS > 0:
        request["max_tokens"] = PORTKEY_MAX_TOKENS
    completion = _get_client().chat.completions.create(**request)
    choice = completion.choices[0]
    content = choice.message.content
    if not content:
        raise RuntimeError("The gateway returned an empty response.")
    usage = getattr(completion, "usage", None)
    diagnostics = {
        "finish_reason": getattr(choice, "finish_reason", None),
        "completion_tokens": getattr(usage, "completion_tokens", None),
        "response_chars": len(content),
    }
    _LAST_LLM_DIAGNOSTICS.clear(); _LAST_LLM_DIAGNOSTICS.update(diagnostics)
    print("  LLM response:",
          f"chars={diagnostics['response_chars']}",
          f"finish_reason={diagnostics['finish_reason']}",
          f"completion_tokens={diagnostics['completion_tokens']}", flush=True)
    # sanitise model id for use in filenames
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", model_id).strip("_")
    return content, safe

print("LLM backend ready (Portkey).")

## 4. Utilities — logging, I/O, iverilog, testbench, parsing, prompts

Runtime logs remain attached to the executing cell, so progress, compiler errors,
testbench results, and detector reports are visible directly in the notebook; no
runtime `.txt` files are created under `logs/`. `iverilog` is not auto-installed
via `apt-get`; on macOS install it with `brew install icarus-verilog`.

In [ ]:
# ── Notebook-native logging ─────────────────────────────────────
def start_logging(log_path=None):
    """Keep stdout/stderr attached to the notebook so all logs appear in-cell."""
    restore_notebook_streams()
    return None, (sys.stdout, sys.stderr)

def stop_logging(logger, original_streams):
    sys.stdout, sys.stderr = original_streams
    if logger is not None:
        logger.close()

# ── File I/O ───────────────────────────────────────────────────
def ensure_dir(path):
    os.makedirs(path, exist_ok=True); return path

def load_designs(base_dir):
    """(design_name, design_code, source_dir) for each standalone *.v and each
    sub-folder whose <folder>/<folder>.v is the top file."""
    base_dir = str(base_dir)
    designs = []
    for fp in glob.glob(os.path.join(base_dir, "*.v")):
        try:
            designs.append((os.path.basename(fp), open(fp).read(), base_dir))
        except Exception as e:
            print(f"  could not read {fp}: {e}")
    for folder in sorted(os.listdir(base_dir)):
        folder_path = os.path.join(base_dir, folder)
        if not os.path.isdir(folder_path):
            continue
        top = os.path.join(folder_path, f"{folder}.v")
        if os.path.exists(top):
            try:
                designs.append((f"{folder}.v", open(top).read(), folder_path))
            except Exception as e:
                print(f"  could not read {top}: {e}")
    print(f"Found {len(designs)} design(s) in {base_dir}")
    return designs

def save_verilog(verilog_code, out_dir, filename):
    last = verilog_code.rfind("endmodule")
    if last != -1:
        verilog_code = verilog_code[:last] + "endmodule"
    path = os.path.join(ensure_dir(out_dir), filename)
    open(path, "w").write(verilog_code.strip())
    return path

def save_metadata(content, out_dir, filename):
    path = os.path.join(ensure_dir(out_dir), filename)
    open(path, "w").write(content.strip())
    return path

def output_dir_for(base_dir, model_name, design_basename):
    return os.path.join(str(base_dir), model_name, design_basename)

# ── iverilog syntax check ──────────────────────────────────────
def require_iverilog():
    if shutil.which("iverilog") is None:
        raise RuntimeError("iverilog not found. Install it (macOS: brew install icarus-verilog).")

def check_verilog_syntax(file_path):
    directory = os.path.dirname(file_path)
    cmd = ["iverilog", "-t", "null", "-Wall", "-I", directory]
    try:
        has_includes = bool(re.search(r'`include\s+["<].+?[">]', open(file_path).read()))
    except Exception:
        has_includes = False
    if not has_includes:
        deps = [f for f in glob.glob(os.path.join(directory, "*.v"))
                if os.path.abspath(f) != os.path.abspath(file_path)]
        cmd.extend(deps)
    cmd.append(file_path)
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
        return (True, "Syntax check passed.") if result.returncode == 0 else (False, result.stderr.strip())
    except subprocess.TimeoutExpired:
        return False, "iverilog timed out."
    except Exception as e:
        return False, f"Error running iverilog: {e}"

# ── Functional testbench (iverilog + vvp) ──────────────────────
def run_testbench(verilog_file, design_name):
    """Returns (True/False, msg), or (None, msg) if no tb_<design>.v exists.
    A testbench should print [PASS]/[FAIL] lines and $finish."""
    base = os.path.splitext(design_name)[0]
    tb_path = os.path.join(str(TESTBENCHES_DIR), f"tb_{base}.v")
    if not os.path.exists(tb_path):
        print(f"  no testbench for {design_name}; skipping functional check.")
        return None, "No testbench found."
    sim_dir = os.path.join(str(WORK_DIR), "sim", base)
    shutil.rmtree(sim_dir, ignore_errors=True)
    os.makedirs(os.path.join(sim_dir, "work"), exist_ok=True)
    sim_out = os.path.join(sim_dir, "sim.out")
    try:
        source_dir = os.path.dirname(verilog_file)
        source_text = open(verilog_file).read()
        has_includes = bool(re.search(r'`include\s+["<].+?[">]', source_text))
        deps = [] if has_includes else [
            f for f in glob.glob(os.path.join(source_dir, "*.v"))
            if os.path.abspath(f) != os.path.abspath(verilog_file)]
        compile_cmd = (["iverilog", "-o", sim_out, "-Wall", "-I", source_dir,
                        "-I", str(TESTBENCHES_DIR)] + deps + [verilog_file, tb_path])
        cr = subprocess.run(compile_cmd, capture_output=True, text=True, timeout=30)
        if cr.returncode != 0:
            return False, f"Testbench compilation failed:\n{cr.stderr.strip()}"
        sr = subprocess.run(["vvp", sim_out], capture_output=True, text=True,
                            timeout=60, cwd=sim_dir)
        out = sr.stdout
        fails = [l for l in out.splitlines() if "[FAIL]" in l]
        passes = [l for l in out.splitlines() if "[PASS]" in l]
        finished = "$finish" in out or sr.returncode == 0
        if not finished:
            return False, f"Simulation did not complete.\nstdout:\n{out[:1000]}\nstderr:\n{sr.stderr[:500]}"
        if fails:
            return False, (f"{len(fails)} test(s) FAILED ({len(passes)} passed).\n" + "\n".join(fails))
        return True, f"Testbench PASSED: {len(passes)} passed, 0 failed."
    except subprocess.TimeoutExpired:
        return False, "Simulation timed out."
    except Exception as e:
        return False, f"Error running testbench: {e}"
    finally:
        shutil.rmtree(sim_dir, ignore_errors=True)

# ── LLM response parsing ───────────────────────────────────────
def parse_llm_response(response_text):
    """Extract (code, explanation, trigger, payload, taxonomy)."""
    header_re = re.compile(
        r'(?im)^\s*(?:#{1,6}\s*)?(?:\*\*)?'
        r'(Code|Explanation|Trigger|Payload|Taxonomy)\s*:\s*(?:\*\*)?\s*$')
    matches = list(header_re.finditer(response_text))
    extracted = {}
    for i, match in enumerate(matches):
        name = match.group(1).title()
        cs = match.end()
        ce = matches[i + 1].start() if i + 1 < len(matches) else len(response_text)
        content = response_text[cs:ce].strip()
        if name == "Code":
            content = re.sub(r"^```[a-z]*\n?", "", content)
            content = re.sub(r"```$", "", content).strip()
            lines = content.split("\n")
            for j, line in enumerate(lines):
                if any(line.strip().startswith(k) for k in ("`timescale", "`include", "module")):
                    content = "\n".join(lines[j:]); break
        extracted[name] = content
    if not extracted.get("Code"):
        fenced = re.search(r'```(?:verilog|systemverilog|v)?\s*\n(.*?)```',
                           response_text, flags=re.IGNORECASE | re.DOTALL)
        if fenced:
            extracted["Code"] = fenced.group(1).strip()
    return (extracted.get("Code", ""), extracted.get("Explanation", ""),
            extracted.get("Trigger", ""), extracted.get("Payload", ""),
            extracted.get("Taxonomy", ""))

_MODULE_DECL_RE = re.compile(r'(?m)^\s*module\s+([A-Za-z_$][\w$]*)\b')

def check_verilog_completeness(verilog_code, reference_code):
    """Return errors when an LLM response ended before the RTL was complete."""
    errors = []
    required_modules = _MODULE_DECL_RE.findall(reference_code or "")
    generated_modules = _MODULE_DECL_RE.findall(verilog_code or "")
    missing = [name for name in required_modules if name not in generated_modules]
    if missing:
        errors.append("missing module(s): " + ", ".join(missing))
    endmodule_count = len(re.findall(r'(?m)^\s*endmodule\b', verilog_code or ""))
    if endmodule_count < len(required_modules):
        errors.append(
            f"only {endmodule_count} endmodule marker(s), expected at least {len(required_modules)}")
    if verilog_code and not re.search(r'endmodule\s*$', verilog_code):
        errors.append("response does not end with endmodule")
    if (verilog_code or "").count("/*") > (verilog_code or "").count("*/"):
        errors.append("unterminated block comment")
    return errors

# ── Prompt builders (verbatim from TrojanGYM) ──────────────────
_RESPONSE_FORMAT = """
Provide a response in EXACTLY this format (one instance of each section):

Code:
<Full modified Verilog code with Trojan comments>

Explanation:
<Two-line description of insertion and impact>

Trigger:
<Two-line description of trigger mechanism>

Payload:
<Two-line description of payload and effects>

Taxonomy:
<Insertion phase: Design
Abstraction level: Register-transfer level
Activation mechanism: [Specify]
Effects: {vulnerability}
Location: [Specify]
Characteristics: [Specify]>
"""

_SYNTH_GUIDELINES = """
Synthesizable Design Guidelines:
1. Preserve the existing reset scheme. Never add or remove reset/clock ports.
2. Use counters for time/event-based triggers.
3. Handle all cases in conditional logic (else/default).
4. Omit non-synthesizable constructs (initial blocks, $display).
5. Only use synthesizable Verilog constructs.
"""

def build_insertion_prompt(design_code, vulnerability, strategy):
    return f"""
Analyze and modify the following Verilog code to insert a hardware Trojan:

<{design_code}>

Task: Insert a "{vulnerability}" Trojan using this strategy: {strategy}
Maintain original functionality. Only ADD internal logic; do not remove anything.
The supplied code is the only file you may modify. Preserve every module port list,
module instance argument list, `include directive, clock/reset scheme, and top-level
interface EXACTLY. In particular, never add a reset port when one did not exist.
Included dependency modules are compiled separately and must not be reproduced or
have their interfaces changed. Include all modules present below. Ensure the design
is fully synthesizable.
{_SYNTH_GUIDELINES}
{_RESPONSE_FORMAT.format(vulnerability=vulnerability)}
CRITICAL: Exactly one of each section. No repetition.
"""

def build_syntax_fix_prompt(broken_code, error_log):
    return f"""
The Verilog compiler detected errors. Fix them while retaining the Trojan.

--- BROKEN CODE ---
{broken_code}

--- COMPILER ERROR LOG ---
{error_log}

--- TASK ---
Return the FIXED, COMPLETE code in the EXACT same format as before:

Code:
<Full fixed Verilog code>

Explanation:
<Brief explanation of what was fixed>

Trigger:
<Same trigger description as before>

Payload:
<Same payload description as before>

Taxonomy:
<Same taxonomy as before>

CRITICAL: You MUST include all five sections. Do not just return raw code.
"""

def build_incomplete_response_prompt(task_prompt, completeness_errors):
    details = "; ".join(completeness_errors)
    return f"""
Your previous answer was truncated ({details}). Start again from the beginning.
Return the COMPLETE response in one answer. Keep comments and prose concise so the
full Verilog fits. Do not stop until every input module has its closing endmodule
and all five requested sections have been emitted.

{task_prompt}
"""

def build_testbench_fix_prompt(verilog_code, tb_failure_log):
    return f"""
The following Verilog design with an inserted hardware Trojan was compiled and
simulated against its testbench, but the simulation reported failures.

Your task is to fix the design so it passes the testbench while KEEPING the
Trojan intact. The testbench tests the original functional behaviour of the
module — your Trojan must not disturb this under normal (non-trigger) conditions.

--- CURRENT VERILOG CODE ---
{verilog_code}

--- TESTBENCH FAILURE LOG ---
{tb_failure_log}

--- TASK ---
Return the FIXED, COMPLETE code in the EXACT same format as before:

Code:
<Full fixed Verilog code>

Explanation:
<Brief explanation of what was fixed>

Trigger:
<Same trigger description as before>

Payload:
<Same payload description as before>

Taxonomy:
<Same taxonomy as before>

CRITICAL: You MUST include all five sections. Do not just return raw code.
"""

def build_evasion_prompt(original_code, aux_files, detected_code, vulnerability,
                         detection_method, detection_report, evasion_strategy,
                         detection_log_content=""):
    aux_section = ""
    for fname, code in (aux_files or {}).items():
        aux_section += f"\nAuxiliary file {fname}:\n{code[:10000]}\n"
    det_log_block = ("Detection Log:\n" + detection_log_content) if detection_log_content else ""
    return f"""
Original clean design:
<MAIN_CODE>
{original_code}
</MAIN_CODE>
{aux_section}
Previous detected design:
{detected_code}

Detection Method: {detection_method}
Detection Report: {detection_report}
{det_log_block}

Task: Re-insert the "{vulnerability}" Trojan into the MAIN_CODE so it evades {detection_method}.
Evasion Strategy: {evasion_strategy}
{_SYNTH_GUIDELINES}
{_RESPONSE_FORMAT.format(vulnerability=vulnerability)}
"""

print("Utilities loaded.")

## 5. Generation pipeline (with syntax + testbench repair)

`generate_with_syntax_check` is the inner loop: LLM → extract code → `iverilog`
syntax check → functional testbench → on failure, feed the error back and retry,
up to `MAX_SYNTAX_RETRIES`.

In [ ]:
def generate_with_syntax_check(prompt, design_name, source_dir, working_dir,
                               max_retries=None):
    """Returns (response_text, model_name, passed)."""
    if max_retries is None:
        max_retries = MAX_SYNTAX_RETRIES
    ensure_dir(working_dir)
    reference_path = os.path.join(source_dir, design_name) if source_dir else ""
    reference_code = (open(reference_path).read()
                      if reference_path and os.path.exists(reference_path) else "")
    if source_dir and os.path.exists(source_dir):
        for fp in glob.glob(os.path.join(source_dir, "*.v")):
            if os.path.basename(fp) != design_name:
                shutil.copy(fp, working_dir)

    temp_file = os.path.join(working_dir, "temp_syntax_check.v")
    current_prompt = prompt
    response_text, model_name = "", "unknown"
    llm_failures = 0

    for attempt in range(max_retries + 1):
        print(f"  attempt {attempt + 1}/{max_retries + 1} ...", flush=True)
        try:
            response_text, model_name = get_llm_response(current_prompt)
        except Exception as e:
            llm_failures += 1
            print(f"  LLM call failed: {e}")
            if "blocked" in str(e).lower():
                return None, None, False
            if llm_failures >= MAX_LLM_CALL_FAILURES:
                print(f"  stopping after {llm_failures} LLM call failures.")
                return None, None, False
            time.sleep(2)
            continue

        verilog_code, *_ = parse_llm_response(response_text)
        if not verilog_code or len(verilog_code.strip()) < 10:
            print("  no code in response; retrying.")
            if attempt < max_retries:
                current_prompt = build_incomplete_response_prompt(
                    current_prompt, ["no parseable Code section"])
            else:
                return None, model_name, False
            continue

        completeness_errors = check_verilog_completeness(verilog_code, reference_code)
        if completeness_errors:
            finish_reason = _LAST_LLM_DIAGNOSTICS.get("finish_reason")
            print("  incomplete LLM response:", "; ".join(completeness_errors),
                  f"(finish_reason={finish_reason})")
            if attempt < max_retries:
                current_prompt = build_incomplete_response_prompt(
                    current_prompt, completeness_errors)
                continue
            print("  max retries reached; incomplete RTL will NOT be saved.")
            return None, model_name, False

        open(temp_file, "w").write(verilog_code)
        passed, message = check_verilog_syntax(temp_file)

        if passed:
            print("  syntax OK")
            tb_passed, tb_msg = run_testbench(temp_file, design_name)
            if tb_passed is None:          # no testbench -> accept
                if os.path.exists(temp_file): os.remove(temp_file)
                return response_text, model_name, True
            if tb_passed:
                print(f"  testbench OK: {tb_msg}")
                if os.path.exists(temp_file): os.remove(temp_file)
                return response_text, model_name, True
            print(f"  testbench FAILED: {tb_msg[:200]}")
            if attempt < max_retries:
                current_prompt = build_testbench_fix_prompt(verilog_code, tb_msg)
            else:
                print("  max retries after testbench failure.")
                if os.path.exists(temp_file): os.remove(temp_file)
                return response_text, model_name, False
        else:
            print(f"  syntax FAILED: {message[:120]}...")
            if attempt < max_retries:
                current_prompt = build_syntax_fix_prompt(verilog_code, message)

    if os.path.exists(temp_file): os.remove(temp_file)
    return response_text, model_name, False


def _save_result(design_name, response_text, model_name, out_dir, vuln_id,
                 attempt, syntax_passed, extra_metadata=None):
    verilog_code, explanation, trigger, payload, taxonomy = parse_llm_response(response_text)
    base = os.path.splitext(design_name)[0]
    folder = output_dir_for(out_dir, model_name, base)
    prefix = "" if syntax_passed else "BROKEN_"
    stem = f"{prefix}{base}_{vuln_id}_{model_name}_A{attempt}"

    v_path = save_verilog(verilog_code, folder, f"{stem}.v")
    print(f"  saved verilog  -> {v_path}")

    meta = [f"Design: {design_name}", f"Vulnerability: {vuln_id}",
            f"Model: {model_name}", f"Attempt: {attempt}",
            f"Syntax passed: {syntax_passed}", "",
            "Explanation:", explanation, "", "Trigger:", trigger, "",
            "Payload:", payload, "", "Taxonomy:", taxonomy]
    if extra_metadata:
        meta += [""] + [f"{k}: {v}" for k, v in extra_metadata.items()]
    meta_path = save_metadata("\n".join(meta), folder, f"{stem}_metadata.txt")
    print(f"  saved metadata -> {meta_path}")
    return v_path, meta_path, verilog_code


def run_generation_pipeline(attempt_number=1):
    """One-shot generation for every design x vulnerability (no detection)."""
    logger, orig = start_logging()
    try:
        require_iverilog()
        designs = load_designs(BASE_DESIGNS_DIR)
        if not designs:
            print(f"No designs in {BASE_DESIGNS_DIR}."); return
        for design_name, design_code, source_dir in designs:
            for vuln_id, vuln_desc in VULNERABILITIES.items():
                print(f"\n{'='*60}\nDesign: {design_name} | {vuln_id} ({vuln_desc})")
                prompt = build_insertion_prompt(design_code, vuln_desc, INSERTION_STRATEGIES[vuln_id])
                base = os.path.splitext(design_name)[0]
                working_dir = os.path.join(str(VULNERABLE_DESIGNS_DIR), "staging", base)
                resp, model_name, passed = generate_with_syntax_check(
                    prompt, design_name, source_dir, working_dir)
                if resp:
                    _save_result(design_name, resp, model_name,
                                 VULNERABLE_DESIGNS_DIR, vuln_id, attempt_number, passed)
                else:
                    print("  generation produced no output.")
    finally:
        stop_logging(logger, orig)

print("Generation pipeline ready.")

## 6. Detection pipeline (Robust-GNN4TJ) — required

This mirrors TrojanGYM's GNN detector and is **mandatory** for the agentic loop in
Section 7. It needs:

* the `Robust_hw2vec` package (`HW2VEC_DIR`), torch + torch-geometric + pyverilog,
* the graph cache `dfg_tj_rtl.pkl` (`DATA_PKL_PATH`),
* the per-HT trained models in `MODEL_PATHS` (each folder has `model.cfg` + `model.pth`).

If any of these is missing, `detect_trojan` / `_ensure_detection_ready` raises
`DetectorUnavailable`, and `run_agentic_loop` aborts immediately (fail-fast) rather
than running without detection. (The generation-only helper
`run_generation_pipeline` in Section 5 does not use the detector and still works.)

Run the install cell below to set up the GNN detector before using the loop.

In [ ]:
# ── GNN detector setup: Python deps + clone/patch Robust_hw2vec ──
# Run once. Heavy (torch, torch-geometric); only executes when you run this cell.
# macOS-friendly: no CUDA index, no apt-get. Install system tools via brew.
import subprocess, sys, glob

def _pip(*pkgs):
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
        return True
    except subprocess.CalledProcessError as e:
        print(f"  pip install failed for {pkgs}: {e}")
        return False

# 1) Python dependencies (CPU/MPS wheels — works on macOS)
_pip("torch", "torchvision", "torchaudio")
_pip("torch_geometric==2.3.1")
_pip("networkx", "scikit-learn", "pyyaml", "tqdm", "pandas", "matplotlib", "seaborn")
_pip("pyverilog", "pydot")  # hw2graph.py imports both; pydot is not preinstalled off-Colab
# pygraphviz needs the system graphviz library + headers FIRST:
#   macOS:  brew install graphviz
#   Debian: sudo apt-get install graphviz libgraphviz-dev pkg-config
_pip("pygraphviz")

# 2) System tools (not auto-installed on macOS)
for _tool, _brew in (("iverilog", "icarus-verilog"), ("dot", "graphviz")):
    if shutil.which(_tool) is None:
        print(f"  [!] '{_tool}' not found -> macOS: brew install {_brew}")

# 3) Clone + patch Robust_hw2vec into HW2VEC_DIR (idempotent)
if not HW2VEC_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/zwangsyc/Robust_hw2vec.git",
                    str(HW2VEC_DIR)], check=True)
    # Patch numpy deprecations + compute_class_weight signature for modern numpy/sklearn
    for fp in glob.glob(str(HW2VEC_DIR / "hw2vec" / "**" / "*.py"), recursive=True):
        text = open(fp).read()
        text = re.sub(r"\bnp\.Inf\b",  "np.inf", text)
        text = re.sub(r"\bnp\.bool\b", "bool",   text)
        text = re.sub(r"\bnp\.int\b",  "int",    text)
        text = re.sub(
            r"compute_class_weight\s*\(\s*['\"]balanced['\"]\s*,\s*np\.unique\s*\(\s*training_labels\s*\)\s*,\s*training_labels\s*\)",
            "compute_class_weight(class_weight='balanced', classes=np.unique(training_labels), y=training_labels)",
            text)
        open(fp, "w").write(text)
    print("hw2vec cloned and patched ->", HW2VEC_DIR)
else:
    print("hw2vec already present ->", HW2VEC_DIR)

# 4) Make hw2vec importable
for _p in (str(HW2VEC_DIR), str(HW2VEC_DIR / "hw2vec")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("Detector setup done.")

In [ ]:
class DetectorUnavailable(RuntimeError):
    pass

_DET = {"cfg": None, "hw2graph": None, "data_proc": None, "models": None}

def _detector_assets_present():
    missing = []
    if not DATA_PKL_PATH.exists():          missing.append(f"pkl: {DATA_PKL_PATH}")
    if not HW2VEC_DIR.exists():             missing.append(f"hw2vec: {HW2VEC_DIR}")
    present_models = [k for k, p in MODEL_PATHS.items() if Path(p).exists()]
    if not present_models:                  missing.append("per-HT model folders")
    return (len(missing) == 0), missing, present_models

def _ensure_detection_ready():
    if _DET["cfg"] is not None:
        return
    ok, missing, _ = _detector_assets_present()
    if not ok:
        raise DetectorUnavailable("Detector assets missing: " + "; ".join(missing))

    for p in (str(HW2VEC_DIR), str(HW2VEC_DIR / "hw2vec")):
        if p not in sys.path:
            sys.path.insert(0, p)

    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from config import Config
    from hw2graph import HW2GRAPH, DataProcessor, heuristic_dfg_from_code
    from graph2vec.models import GRAPH2VEC, GRAPH_CONV, GRAPH_POOL, GRAPH_READOUT

    globals().update(dict(_torch=torch, _nn=nn, _F=F, _DataLoaderImported=True))
    from torch_geometric.loader import DataLoader
    globals()["_TGDataLoader"] = DataLoader
    globals()["_GRAPH2VEC"] = GRAPH2VEC
    globals()["_GRAPH_CONV"] = GRAPH_CONV
    globals()["_GRAPH_POOL"] = GRAPH_POOL
    globals()["_GRAPH_READOUT"] = GRAPH_READOUT
    globals()["_heuristic_dfg_from_code"] = heuristic_dfg_from_code

    args = ["--yaml_path", str(YAML_PATH), "--data_pkl_path", str(DATA_PKL_PATH),
            "--graph_type", "DFG", "--model_path", ""]
    cfg = Config(args)
    data_proc = DataProcessor(cfg)
    data_proc.read_graph_data_from_cache(cfg.data_pkl_path)
    hw2graph = HW2GRAPH(cfg)

    models = {}
    for name, path in MODEL_PATHS.items():
        path = Path(path)
        if not path.exists():
            print(f"  {name} model NOT found at {path}"); continue
        model = GRAPH2VEC(cfg)
        with open(path / "model.cfg") as f:
            mcfg = json.load(f)
        model.set_graph_conv([GRAPH_CONV(*c) for c in mcfg["convs"]])
        model.set_graph_pool(GRAPH_POOL(*mcfg["pool"]))
        model.set_graph_readout(GRAPH_READOUT(mcfg["readout"]))
        model.set_output_layer(nn.Linear(*mcfg["fc"]))
        model.load_state_dict(torch.load(str(path / "model.pth"), map_location="cpu"))
        model.to(cfg.device).eval()
        models[name] = model
        print(f"  {name} model loaded")

    if not models:
        raise DetectorUnavailable("No per-HT models could be loaded.")
    _DET.update(cfg=cfg, hw2graph=hw2graph, data_proc=data_proc, models=models)

def _copy_included_sources(verilog_file, tmp):
    """Copy the top file and every `include dependency into the detector sandbox."""
    source = Path(verilog_file).resolve()
    tmp = Path(tmp)
    include_pattern = r'^\s*`include\s+"[^"]+"\s*$'
    (tmp / source.name).write_text(
        re.sub(include_pattern, "", source.read_text(errors="replace"),
               flags=re.MULTILINE))
    pending, copied = [source], {source.name}
    while pending:
        current = pending.pop()
        text = current.read_text(errors="replace")
        for include_name in re.findall(r'`include\s+"([^"]+)"', text):
            include_base = Path(include_name).name
            if include_base in copied:
                continue
            candidates = [current.parent / include_name, source.parent / include_name]
            candidates.extend(BASE_DESIGNS_DIR.rglob(include_base))
            dependency = next((p for p in candidates if p.is_file()), None)
            if dependency is None:
                raise DetectorUnavailable(
                    f"Missing Verilog include '{include_name}' required by {current}")
            destination = tmp / include_name
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_text(
                re.sub(include_pattern, "", dependency.read_text(errors="replace"),
                       flags=re.MULTILINE))
            copied.add(include_base)
            pending.append(dependency)

def _design_to_loader(verilog_file):
    tmp = os.path.join(str(WORK_DIR), "hw2vec_test_design")
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    _copy_included_sources(verilog_file, tmp)
    # PyVerilog exact DFG expansion is pathological on AES S-box case tables.
    # Detect them up front and use the same deterministic fallback immediately.
    verilog_sources = list(Path(tmp).glob("*.v"))
    lookup_entries = sum(len(re.findall(
        r"(?im)^\s*8'h[0-9a-f]{2}\s*:", p.read_text(errors="replace")))
        for p in verilog_sources)
    force_exact = os.environ.get("TJGYM_FORCE_EXACT_DFG", "0") == "1"
    use_fast_dfg = lookup_entries >= 128 and not force_exact
    if use_fast_dfg:
        code_path = _DET["hw2graph"].preprocess(Path(tmp))
        code = Path(code_path).read_text(errors="replace")
        print(f"[HW2GRAPH] Large lookup table detected ({lookup_entries} entries); "
              "using heuristic DFG directly.")
        graph = _heuristic_dfg_from_code(code, name="snippet", hw_type="DFG")
    else:
        # Bound exact extraction when configured; HW2GRAPH catches TimeoutError and
        # falls back to its heuristic DFG. Zero means no timeout.
        import signal
        can_alarm = hasattr(signal, "SIGALRM") and DFG_TIMEOUT_SECONDS > 0
        old_handler = None
        if can_alarm:
            old_handler = signal.signal(
                signal.SIGALRM,
                lambda *_: (_ for _ in ()).throw(TimeoutError(
                    f"exact DFG exceeded {DFG_TIMEOUT_SECONDS}s")))
            signal.setitimer(signal.ITIMER_REAL, DFG_TIMEOUT_SECONDS)
        try:
            graph = _DET["hw2graph"].code2graph(Path(tmp))
        finally:
            if can_alarm:
                signal.setitimer(signal.ITIMER_REAL, 0)
                signal.signal(signal.SIGALRM, old_handler)
    graph.graph["hw_type"] = "Test"; graph.graph["label"] = 0
    _DET["data_proc"].process(graph)
    test_data = _DET["data_proc"].get_graphs()[-1]
    return _TGDataLoader([test_data], shuffle=False, batch_size=1)

def _run_inference(model, loader, device):
    outputs = []
    with _torch.no_grad():
        for data in loader:
            data.to(device)
            out, _ = model.embed_graph(data.x, data.edge_index, data.batch)
            out = _F.log_softmax(model.mlp(out), dim=1)
            outputs.append(out.cpu())
    outputs = _torch.cat(outputs).reshape(-1, 2).detach()
    pred = outputs.max(1)[1].item()
    probs = _torch.exp(outputs).detach().numpy()[0]
    return pred, probs

def detect_trojan(verilog_file):
    """Run the GNN ensemble. Raises DetectorUnavailable if assets missing."""
    _ensure_detection_ready()
    loader = _design_to_loader(verilog_file)
    results, triggered = {}, []
    for name, model in _DET["models"].items():
        pred, probs = _run_inference(model, loader, _DET["cfg"].device)
        results[name] = {"prediction": pred, "prob_clean": float(probs[0]),
                         "prob_trojan": float(probs[1])}
        if pred == 1:
            triggered.append(name)
    trojan_probs = [m["prob_trojan"] for m in results.values()]
    ensemble = {
        "trojan_votes": len(triggered),
        "model_count": len(results),
        "mean_prob_trojan": (sum(trojan_probs) / len(trojan_probs)
                              if trojan_probs else 0.0),
        "max_prob_trojan": max(trojan_probs, default=0.0),
        "min_prob_trojan": min(trojan_probs, default=0.0),
    }
    return {"file": verilog_file, "per_model": results,
            "triggered": triggered,
            "ensemble": ensemble,
            "verdict": "TROJAN" if triggered else "CLEAN"}

def print_detection_report(r):
    print(f"\nGNN prediction report\nFile: {r['file']}")
    print("-" * 64)
    print(f"{'Model':<8} {'Prediction':<12} {'P(clean)':>12} {'P(trojan)':>12}")
    print("-" * 64)
    for name, m in r["per_model"].items():
        label = "TROJAN" if m["prediction"] == 1 else "CLEAN"
        print(f"{name:<8} {label:<12} {m['prob_clean']:>12.6f} "
              f"{m['prob_trojan']:>12.6f}")
    print("-" * 64)
    e = r["ensemble"]
    print(f"Ensemble verdict : {r['verdict']}")
    print(f"Trojan votes     : {e['trojan_votes']}/{e['model_count']}")
    print(f"Mean P(trojan)   : {e['mean_prob_trojan']:.6f}")
    print(f"Max P(trojan)    : {e['max_prob_trojan']:.6f}")
    print(f"Min P(trojan)    : {e['min_prob_trojan']:.6f}")
    if r["triggered"]:
        print("Triggered models  :", ", ".join(r["triggered"]))

_ok, _missing, _present = _detector_assets_present()
if _ok:
    print("Detector assets present.")
else:
    print("Detector assets MISSING (required for run_agentic_loop):")
    for _m in _missing:
        print("  -", _m)
    print("Configure these before running the agentic loop, or it will raise DetectorUnavailable.")
if _present:
    print("Models found:", ", ".join(_present))

## 7. Agentic feedback loop — generate → detect → evade

This is the core of TrojanGYM. For each design × Trojan type it runs the initial
insertion, then up to `MAX_EVASION_ATTEMPTS` detect→evade cycles, saving every
intermediate design, its metadata, and its detection log.

The GNN detector is **required**: `run_agentic_loop` calls `_ensure_detection_ready()`
up front and raises `DetectorUnavailable` (aborting the run) if the detector assets
from Section 6 are not configured.

In [ ]:
def _load_design_with_aux(design_name, source_dir):
    main_path = os.path.join(source_dir, design_name)
    if not os.path.exists(main_path):
        return None, {}
    main_code = open(main_path).read()
    aux = {os.path.basename(fp): open(fp).read()
           for fp in glob.glob(os.path.join(source_dir, "*.v"))
           if os.path.basename(fp) != design_name}
    return main_code, aux


def run_agentic_loop(detection_key="D4", evasion_strategy_key="S4",
                     max_evasion_attempts=None, design_names=None,
                     vulnerability_ids=None):
    """Run the loop, optionally filtering designs and Trojan types.

    Example: design_names={"aes_128.v"}, vulnerability_ids={"T1"}.
    """
    if max_evasion_attempts is None:
        max_evasion_attempts = MAX_EVASION_ATTEMPTS
    logger, orig = start_logging()
    detection_method = DETECTION_METHODS[detection_key]
    evasion_strategy = EVASION_STRATEGIES[evasion_strategy_key]

    try:
        # Detector is REQUIRED: fail fast without leaving stdout redirected.
        _ensure_detection_ready()
        require_iverilog()
        designs = load_designs(BASE_DESIGNS_DIR)
        if design_names is not None:
            wanted = {design_names} if isinstance(design_names, str) else set(design_names)
            designs = [d for d in designs if d[0] in wanted]
        if not designs:
            raise ValueError(f"No matching designs in {BASE_DESIGNS_DIR}.")
        if vulnerability_ids is None:
            selected_vulnerabilities = VULNERABILITIES.items()
        else:
            wanted_vulns = ({vulnerability_ids} if isinstance(vulnerability_ids, str)
                            else set(vulnerability_ids))
            unknown = wanted_vulns - set(VULNERABILITIES)
            if unknown:
                raise ValueError(f"Unknown vulnerability IDs: {sorted(unknown)}")
            selected_vulnerabilities = [(k, VULNERABILITIES[k])
                                        for k in VULNERABILITIES if k in wanted_vulns]

        for design_name, design_code, source_dir in designs:
            for vuln_id, vuln_desc in selected_vulnerabilities:
                print(f"\n{'='*65}\n  {design_name} | {vuln_id}: {vuln_desc}\n{'='*65}")
                last_v_path = None
                detected = False

                for attempt in range(1, max_evasion_attempts + 2):
                    print(f"\n--- Attempt A{attempt} ---")

                    if attempt == 1:
                        prompt = build_insertion_prompt(
                            design_code, vuln_desc, INSERTION_STRATEGIES[vuln_id])
                    else:
                        if not detected:
                            print("  previous attempt CLEAN — no evasion needed."); break
                        orig_code, aux = _load_design_with_aux(design_name, source_dir)
                        det_code = open(last_v_path).read() if last_v_path else ""
                        det_log = ""
                        det_log_path = last_v_path.replace(".v", "_detection.txt") if last_v_path else ""
                        if det_log_path and os.path.exists(det_log_path):
                            det_log = open(det_log_path).read()
                        prompt = build_evasion_prompt(
                            orig_code, aux, det_code, vuln_desc, detection_method,
                            f"A{attempt-1} was detected as TROJAN.", evasion_strategy, det_log)

                    base = os.path.splitext(design_name)[0]
                    working_dir = os.path.join(str(VULNERABLE_DESIGNS_DIR), "staging", base)
                    resp, model_name, syntax_passed = generate_with_syntax_check(
                        prompt, design_name, source_dir, working_dir)
                    if not resp:
                        print("  generation failed entirely."); break

                    extra = ({"evasion_attempt": attempt, "detection_method": detection_method}
                             if attempt > 1 else None)
                    v_path, _, _ = _save_result(
                        design_name, resp, model_name, VULNERABLE_DESIGNS_DIR,
                        vuln_id, attempt, syntax_passed, extra_metadata=extra)
                    last_v_path = v_path

                    if not syntax_passed:
                        print("  saved with known syntax errors; detection skipped."); break

                    print(f"\n  detecting {os.path.basename(v_path)} ...")
                    try:
                        det = detect_trojan(v_path)
                        print_detection_report(det)
                        detected = (det["verdict"] == "TROJAN")
                        e = det["ensemble"]
                        det_lines = [
                            f"File: {v_path}",
                            f"Verdict: {det['verdict']}",
                            f"Trojan votes: {e['trojan_votes']}/{e['model_count']}",
                            f"Mean P(trojan): {e['mean_prob_trojan']:.6f}",
                            f"Max P(trojan): {e['max_prob_trojan']:.6f}",
                            f"Min P(trojan): {e['min_prob_trojan']:.6f}",
                            "", "Per-model predictions:",
                        ]
                        for mn, m in det["per_model"].items():
                            label = "TROJAN" if m["prediction"] == 1 else "CLEAN"
                            det_lines.append(
                                f"{mn}: prediction={label} pred={m['prediction']} "
                                f"P(clean)={m['prob_clean']:.6f} "
                                f"P(trojan)={m['prob_trojan']:.6f}")
                        save_metadata("\n".join(det_lines), os.path.dirname(v_path),
                                      os.path.basename(v_path).replace(".v", "_detection.txt"))
                    except Exception as e:
                        print(f"  detection failed: {e}"); detected = False; break

                    if not detected:
                        print(f"  A{attempt} evaded detection. Stopping evasion loop."); break
                    if attempt == max_evasion_attempts + 1:
                        print(f"  max evasion attempts reached for {design_name}/{vuln_id}.")
    finally:
        stop_logging(logger, orig)

print("Agentic loop ready.")

## 8. Load designs

Drop your clean, synthesizable `.v` files into `BASE_DESIGNS_DIR`, or one project
per sub-folder (`<name>/<name>.v` as the top module, with its dependencies beside
it). Optional functional checks: add `tb_<name>.v` to `TESTBENCHES_DIR`, printing
`[PASS]`/`[FAIL]` lines and calling `$finish`.

Use the generic helper below to pull `.v` files from any folder you choose into
`BASE_DESIGNS_DIR`.

In [ ]:
def import_designs_from(src_folder):
    """Copy every *.v found under src_folder into BASE_DESIGNS_DIR."""
    src_root = Path(src_folder)
    if not src_root.exists():
        print(f"Not found: {src_root}"); return
    found = list(src_root.rglob("*.v"))
    for fp in found:
        shutil.copy2(fp, BASE_DESIGNS_DIR / fp.name)
    print(f"Copied {len(found)} .v file(s) into {BASE_DESIGNS_DIR}")
    return load_designs(BASE_DESIGNS_DIR)

# import_designs_from("/path/to/your/clean_rtl")
print("Place designs in:", BASE_DESIGNS_DIR)

## 9. Run

The active smoke call below sends the AES RTL to the configured Portkey gateway and
may incur usage charges when this cell is executed. Merely opening the notebook does
not execute it. Comment it out when you only want to run a different entry point.

In [ ]:
# ── Full agentic loop: generate -> validate (syntax+testbench) -> detect -> evade ──
# Prerequisites: PORTKEY_API_KEY set; Sections 2-7 cells already run; designs present
# in BASE_DESIGNS_DIR; GNN detector configured (Section 6). This makes paid Portkey
# calls (one per design x Trojan type x attempt). This smoke run exercises AES/T1
# once; set max_evasion_attempts higher to enable detector-guided evasion retries.
restore_notebook_streams()
print("[TrojanGYM] Starting AES/T1 smoke run; progress will appear below.", flush=True)
run_agentic_loop(detection_key="D4", evasion_strategy_key="S4",
                   design_names={"aes_128.v"}, vulnerability_ids={"T1"},
                   max_evasion_attempts=0)
print("[TrojanGYM] AES/T1 smoke run completed.", flush=True)

# ── Other entry points — uncomment ONE at a time (keep the loop above commented
#    if you use these) ──────────────────────────────────────────────────────────
#
# Full AES run for all T1-T4 types with detector-guided evasion:
#   run_agentic_loop(design_names={"aes_128.v"})
#
# One-shot generation only, no detection feedback:
#   run_generation_pipeline(attempt_number=1)
#
# Detect a single EXISTING generated design (finds the newest; needs >=1 result):
#   import glob
#   _f = sorted(glob.glob(str(VULNERABLE_DESIGNS_DIR / "**" / "*.v"), recursive=True))
#   assert _f, "No generated designs yet — run the loop or run_generation_pipeline first."
#   print_detection_report(detect_trojan(_f[-1]))